# Retail Inventory Management System
## Notebook 2 — Silver Layer
### Purpose
This notebook reads raw data from the Bronze layer, applies data cleaning and transformation,
and loads it into Silver Delta tables using incremental MERGE logic.
Only new or changed products are processed — unchanged rows are skipped automatically.

In [0]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("silver_notebook")

logger.info("Silver pipeline started")

2026-04-07 09:47:06,425 - INFO - Silver pipeline started


### Importing required libraries

In [0]:

spark.sql("USE CATALOG retail_inventory")

import logging
from pyspark.sql.functions import col, current_date, current_timestamp, explode, array, lit
from delta.tables import DeltaTable


#### Read from Bronze Layer

In [0]:
#  read from bronze
bronze_df = spark.table("retail_inventory.bronze.raw_products")
print(f"Rows coming from bronze: {bronze_df.count()}")

#  clean and select required columns
silver_clean = bronze_df.select(
    col("id").cast("int"),
    col("title").alias("product_name"),
    col("category"),
    col("brand"),
    col("price").cast("double"),
    col("stock").cast("int"),
    col("rating").cast("double"),
    current_date().alias("load_date"),
    current_timestamp().alias("updated_at")
).dropna(subset=["id"]) \
 .dropDuplicates(["id"])

print(f"Rows after cleaning: {silver_clean.count()}")

2026-04-07 09:47:07,120 - INFO - Received command c on object id p0
2026-04-07 09:47:07,312 - INFO - Received command c on object id p0


Rows coming from bronze: 30


2026-04-07 09:47:07,599 - INFO - Received command c on object id p0


Rows after cleaning: 30


####  Incremental Load using MERGE

In [0]:
from delta.tables import DeltaTable

try:
    logger.info("Starting merge operation into Silver table")

    if not spark.catalog.tableExists("retail_inventory.silver.products"):
        logger.info("Silver table does not exist. Creating table for the first time")

        silver_clean.write \
            .format("delta") \
            .saveAsTable("retail_inventory.silver.products")

        logger.info("Silver table created and initial data loaded")

    else:
        logger.info("Silver table exists. Starting merge (incremental load)")

        silver_table = DeltaTable.forName(
            spark, "retail_inventory.silver.products"
        )

        (
            silver_table.alias("existing")
            .merge(
                silver_clean.alias("incoming"),
                "existing.id = incoming.id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        logger.info("Merge completed successfully (Upsert done)")

except Exception as e:
    logger.error(f"Merge operation failed: {str(e)}")
    raise

2026-04-07 09:47:07,745 - INFO - Starting merge operation into Silver table
2026-04-07 09:47:07,898 - INFO - Silver table exists. Starting merge (incremental load)
2026-04-07 09:47:08,313 - INFO - Received command c on object id p0
2026-04-07 09:47:08,600 - INFO - Received command c on object id p0
2026-04-07 09:47:09,254 - INFO - Received command c on object id p0
2026-04-07 09:47:09,312 - INFO - Received command c on object id p0
2026-04-07 09:47:10,169 - INFO - Received command c on object id p0
2026-04-07 09:47:10,312 - INFO - Received command c on object id p0
2026-04-07 09:47:10,823 - INFO - Merge completed successfully (Upsert done)


#### create inventory_by_store Table

In [0]:
from pyspark.sql.functions import explode, array, lit, round, rand, when, col, current_date, struct

products_df = spark.table("retail_inventory.silver.products")

store1 = products_df.withColumn("store_id", lit(1)) \
                    .withColumn("store_stock", col("stock"))

store2 = products_df.withColumn("store_id", lit(2)) \
                    .withColumn("store_stock", 
                        (col("stock") * 0.6).cast("int"))

store3 = products_df.withColumn("store_id", lit(3)) \
                    .withColumn("store_stock", 
                        (col("stock") * 0.3).cast("int"))

inventory_df = store1.union(store2).union(store3) \
    .select(
        col("id"),
        col("product_name"),
        col("category"),
        col("brand"),
        col("price"),
        col("store_id"),
        col("store_stock").alias("stock"),
        lit(10).alias("low_stock_threshold"),
        lit(80).alias("overstock_threshold"),
        current_date().alias("load_date")
    ).orderBy("id", "store_id")

inventory_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_inventory.silver.inventory_by_store")

print(f"Inventory by store rows: {inventory_df.count()}")

2026-04-07 09:47:11,313 - INFO - Received command c on object id p0
2026-04-07 09:47:12,250 - INFO - Received command c on object id p0
2026-04-07 09:47:12,251 - INFO - Received command c on object id p0
2026-04-07 09:47:12,252 - INFO - Received command c on object id p0
2026-04-07 09:47:12,313 - INFO - Received command c on object id p0
2026-04-07 09:47:13,139 - INFO - Received command c on object id p0
2026-04-07 09:47:13,140 - INFO - Received command c on object id p0
2026-04-07 09:47:13,141 - INFO - Received command c on object id p0


Inventory by store rows: 90


#### Verify  inventory_by_store Table

In [0]:
%sql
select* from retail_inventory.silver.inventory_by_store
limit 5;

id,product_name,category,brand,price,stock,rating,load_date,updated_at,store_id,low_stock_threshold,overstock_threshold
1,Essence Mascara Lash Princess,beauty,Essence,9.99,99,null,2026-04-07,null,1,10,80
1,Essence Mascara Lash Princess,beauty,Essence,9.99,59,null,2026-04-07,null,2,10,80
1,Essence Mascara Lash Princess,beauty,Essence,9.99,29,null,2026-04-07,null,3,10,80
2,Eyeshadow Palette with Mirror,beauty,Glamour Beauty,19.99,34,null,2026-04-07,null,1,10,80
2,Eyeshadow Palette with Mirror,beauty,Glamour Beauty,19.99,20,null,2026-04-07,null,2,10,80


In [0]:
logger.info("Silver pipeline completed successfully")

2026-04-07 09:47:14,056 - INFO - Silver pipeline completed successfully
